# 03 · Leaderboard on the held-out year 2023, independent Argo check, and export
Every method is scored on the same year (never used in training), grid, depths and Argo profiles.
GLORYS and HYCOM are scored against the same floats, so the Argo column compares like with like.

In [ ]:
# ── Setup: Drive (data) + GitHub (code) + deps ──────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/OceanEmbed'          # data, checkpoints, outputs live here
REPO = '/content/Sylithe_Ocean_Embedding'
import os, sys; os.makedirs(ROOT, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/karan27-dev/Sylithe_Ocean_Embedding {REPO}
sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m.startswith('oceanembed')]:
    del sys.modules[m]                              # re-running this cell picks up freshly pulled code
!git -C {REPO} log --oneline -1
# Install only what this Colab image lacks (images differ between sessions), one package at a time, so pip
# never searches a large version space (that is what hung before).
import importlib, importlib.util, subprocess
NEED = {'xarray': 'xarray', 'zarr': 'zarr', 'dask': 'dask', 'netCDF4': 'netCDF4', 'scipy': 'scipy',
        'ee': 'earthengine-api', 'copernicusmarine': 'copernicusmarine', 'earthaccess': 'earthaccess', 'psutil': 'psutil'}
for mod, pkg in NEED.items():
    if importlib.util.find_spec(mod) is None:
        print('installing', pkg)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', pkg], check=True)
importlib.invalidate_caches()
for mod in NEED:
    print(f'{mod:17s}', getattr(importlib.import_module(mod), '__version__', 'ok'))

In [ ]:
import glob, shutil, torch, numpy as np, pandas as pd, xarray as xr
from oceanembed import config as C, dataset as D, train as TR, infer as I, baselines as B, benchmark as BM
for s in ['inputs.zarr', 'target.zarr', 'hycom.zarr']:
    if os.path.exists(f'{ROOT}/{s}') and not os.path.exists(f'/content/{s}'):
        print('copying', s); shutil.copytree(f'{ROOT}/{s}', f'/content/{s}')
S = D.load_stats(f'{ROOT}/stats_v2.npz')
cfg = C.TrainConfig(); CK = f'{ROOT}/checkpoints/oceanembed_w{cfg.window}'
G = xr.open_zarr('/content/target.zarr').thetao.sel(time=slice(*C.TEST))
prof = pd.read_parquet(f'{ROOT}/argo_2023.parquet'); print(len(prof), 'Argo profiles')
lb = BM.Leaderboard(G, prof)
def load(ckpt, c):
    _, m = TR.new_models(c); return TR.load_weights(m, ckpt)

In [ ]:
# OceanEmbed: every trained seed (ensemble) and the first seed alone
seeds = sorted(glob.glob(f'{CK}/seed*/glorys_best.pt')); print('seeds:', seeds)
members = [load(p, cfg) for p in seeds]
rec = I.reconstruct(members, '/content/inputs.zarr', S, *C.TEST, window=cfg.window)
rec.to_netcdf(f'{ROOT}/OceanEmbed_NIO_T_2023.nc')                     # the PS deliverable (with σ)
lb.add(f'OceanEmbed ({len(members)}-model ensemble)', 'ours', rec.thetao)
if len(members) > 1:
    lb.add('OceanEmbed (single model)', 'ours', I.reconstruct(members[0], '/content/inputs.zarr', S, *C.TEST, window=cfg.window).thetao)

In [ ]:
# The published method, retrained on our data; then the baselines; then reference products vs Argo
cfg3 = C.TrainConfig(arch='attn_unetpp3d', base=16, batch_size=2)
p3 = f'{ROOT}/checkpoints/attn_unetpp3d_w{cfg3.window}/glorys_best.pt'
if os.path.exists(p3):
    lb.add('Attention 3D U-Net++ (Wang et al. 2026), retrained here', 'published method (retrained here)',
           I.reconstruct(load(p3, cfg3), '/content/inputs.zarr', S, *C.TEST, window=cfg3.window).thetao)
lb.add(B.Ridge.name, 'baseline', I.reconstruct_baseline(B.Ridge(S).load(f'{ROOT}/baselines/ridge.npz'),
       '/content/inputs.zarr', S, *C.TEST, window=cfg.window).thetao)
lb.add(B.Climatology.name, 'baseline', I.reconstruct_baseline(B.Climatology(S), '/content/inputs.zarr', S, *C.TEST,
       window=cfg.window).thetao)
lb.add('GLORYS12 reanalysis (the training target)', 'reference product', G, vs_glorys=False)
if os.path.exists('/content/hycom.zarr'):
    lb.add('HYCOM GOFS 3.1 (independent model, via GEE)', 'reference product', xr.open_zarr('/content/hycom.zarr').thetao)
summary = lb.save(f'{ROOT}/leaderboard'); summary.round(3)

In [ ]:
# Is the predicted uncertainty honest? Fraction of Argo errors inside ±1σ should be ≈ 68 %
m = BM.A.match(BM.A.match(prof, rec.thetao, 'ours'), rec.thetao_sigma, 'sig')
inside = [(np.abs(m[f'ours_T{d}'] - m[f'T{d}']) <= m[f'sig_T{d}']).mean() for d in C.DEPTHS]
pd.Series(inside, index=C.DEPTHS, name='coverage@1σ').round(2)

In [ ]:
# Export for the web console (format v2, oceanembed/webexport.py): maps for the Cyclone Mocha window,
# Argo scores, σ calibration and the embedding over the test year. Copy the folder's contents into web/public/data/.
from oceanembed import webexport as W
days = pd.date_range('2023-05-01', '2023-05-31')   # pre-monsoon / Cyclone Mocha window
W.export_model(f'{ROOT}/web_export', rec, G, '/content/inputs.zarr', S, members, cfg.window, days, profiles=prof,
               hycom=xr.open_zarr('/content/hycom.zarr').thetao if os.path.exists('/content/hycom.zarr') else None,
               leaderboard_json=f'{ROOT}/leaderboard/leaderboard.json')